In [ ]:
#致正在看这个notebook的导师：这个算是我一开始学习深度学习时精心打磨出来的一个作品，期间借这道题学习到了很多知识，我当时的成果目前还无人超越（得意）
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import astropy.io.fits as pyfits
from scipy.ndimage import gaussian_filter

In [2]:
DATA_DIR = "/bohr/train-t3g1/v1"
DEVICE = "cuda"
BATCH_SIZE = 4
SMOOTHING = 2

In [3]:
class AstroDataset(Dataset):
    def __init__(self, map_paths, cat_paths=None):
        self.map_paths = map_paths
        self.cat_paths = cat_paths
    def __len__(self):
        return len(self.map_paths)
    def __getitem__(self, idx):
        map_data = gaussian_filter(pyfits.open(self.map_paths[idx])[0].data, sigma=SMOOTHING) 
        if self.cat_paths is None:
            return torch.FloatTensor(map_data).unsqueeze(0)
        cat = pyfits.open(self.cat_paths[idx])[0].data
        target = np.zeros((1024, 1024))
        for y, x in cat[:, 1:3]:
            target[int(y), int(x)] = 1.0
        return torch.FloatTensor(map_data).unsqueeze(0), torch.FloatTensor(target), self.cat_paths[idx]

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1,bias=False),  # 输入通道1，输出通道16
            nn.BatchNorm2d(32),  # 批量归一化
            nn.ReLU(),
            nn.Dropout(0.1),  # dropout增加泛化能力
            nn.Conv2d(32, 16, kernel_size=3, padding=1,bias=False),  # 输入通道32，输出通道32
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 4, kernel_size=3, padding=1,bias=False),  # 输入通道64，输出通道32
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.Conv2d(4, 1, kernel_size=1),  # 输入通道32，输出通道1
            

            nn.Sigmoid()
        )
        
        

    def forward(self, x):
        x = self.net(x)
        return x
model = Model().to('cuda')
model.load_state_dict(torch.load("/bohr/model-dpdp/v5/phy_model3.pt", map_location="cuda"))
model.train(False)

In [5]:
import zipfile
if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get('DATA_PATH') + "/"
    TEST_DATA_DIR = DATA_PATH + 'halo_testA'
    test_size = len(os.listdir(os.path.join(TEST_DATA_DIR, 'map')))
    dataset = AstroDataset([os.path.join(TEST_DATA_DIR, f'map/{i}.fits') for i in range(1, test_size+1)])
    loader = DataLoader(dataset, batch_size=test_size, shuffle=False)
    model.eval()
    for images in loader:
        with torch.no_grad():
            outputs = model(images.to(DEVICE)).cpu().numpy().squeeze(1)
        np.save('submissionsA.npy', outputs)
    TEST_DATA_DIR = DATA_PATH + 'halo_testB'
    test_size = len(os.listdir(os.path.join(TEST_DATA_DIR, 'map')))
    dataset = AstroDataset([os.path.join(TEST_DATA_DIR, f'map/{i}.fits') for i in range(1, test_size+1)])
    loader = DataLoader(dataset, batch_size=test_size, shuffle=False)
    model.eval()
    for images in loader:
        with torch.no_grad():
            outputs = model(images.to(DEVICE)).cpu().numpy().squeeze(1)
        np.save('submissionsB.npy', outputs)
    with zipfile.ZipFile('submission.zip', 'w') as zipf:
        zipf.write('submissionsA.npy')
        zipf.write('submissionsB.npy')